# OTT Enricher (SerpAPI AI Overview)

This notebook enriches **recent movies (last 2 months)** with OTT platform and OTT release date using **SerpAPI's `ai_overview`**.

## What it does
- Loads your Excel master file
- Filters to movies whose `releaseDate` is within the last **2 months**
- Enriches only rows missing OTT info (`ottList` empty / `[]` / NaN)
- Uses **1 SerpAPI call per movie** (keeps you under monthly limits)
- Writes an enriched Excel file


In [147]:
import re
import time
import json
from datetime import datetime
from dateutil.relativedelta import relativedelta
from dateutil import parser as dtparser
import pandas as pd
import requests


In [148]:
# ====== CONFIG ======
SERP_API_KEY = "e1a2fa219eb0e18f4b3c7d8c0a665d4346e5d8992fed6aff6234a7fb3e65d6f1"

# Input/Output
INPUT_XLSX = "movies_master.xlsx"   # change if needed
OUTPUT_XLSX = "movies_master_enriched_recent.xlsx"

# Columns (adjust if your sheet uses different names)
COL_TITLE = "title"
COL_LANGUAGE = "language"
COL_RELEASEDATE = "releaseDate"   # theatrical/release date column used to filter recency
COL_STATUS = "status"             # optional

COL_OTT_LIST = "ottList"
COL_OTT_DATE = "ottReleaseDate"   # will be created if missing
COL_CONFIDENCE = "ottConfidence"  # will be created if missing
COL_EVIDENCE = "ottEvidence"      # will be created if missing
COL_SOURCES = "ottSources"        # will be created if missing

# Rate limiting
SLEEP_SECONDS = 1.2   # polite pause between SerpAPI calls
MAX_CALLS = 240       # safety cap below 250


In [149]:
PLATFORMS = {
    "Netflix": ["netflix"],
    "Prime Video": ["prime video", "amazon prime", "amazon prime video"],
    "Disney+ Hotstar": ["hotstar", "disney+ hotstar", "disney hotstar"],
    "ZEE5": ["zee5", "zee 5"],
    "SonyLIV": ["sonyliv", "sony liv"],
    "JioCinema": ["jiocinema", "jio cinema"],
    "Aha": ["aha"],
    "Sun NXT": ["sun nxt", "sunnxt"],
    "YouTube": ["youtube"],
}

DATE_PATTERNS = [
    # e.g., January 23, 2026
    r"(Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+\d{1,2},?\s+\d{4}",
    # e.g., 23 January 2026
    r"\d{1,2}\s+(Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\s+\d{4}",
    # e.g., 2026-01-23
    r"\b\d{4}-\d{2}-\d{2}\b",
    # fuzzy: second week of January 2026
    r"\b(first|second|third|fourth)\s+week\s+of\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}\b",
]

OTT_CONTEXT = re.compile(r"\b(ott|streaming|digital premiere|premiere|now streaming|available to watch|watch on|stream on)\b", re.I)

def is_missing_ott(val) -> bool:
    if val is None:
        return True
    if isinstance(val, float) and pd.isna(val):
        return True
    if isinstance(val, list):
        return len(val) == 0
    s = str(val).strip().lower()
    return s in ("", "nan", "none", "null", "[]", "[ ]", "{}")

def extract_platforms(text: str):
    t = text.lower()
    found = []
    for canon, keys in PLATFORMS.items():
        if any(k in t for k in keys):
            found.append(canon)
    return sorted(set(found))

def extract_ott_date(text: str):
    # Prefer dates near OTT/streaming context
    for m in OTT_CONTEXT.finditer(text):
        start = max(0, m.start() - 350)
        end = min(len(text), m.end() + 350)
        window = text[start:end]
        for pat in DATE_PATTERNS:
            dm = re.search(pat, window, re.I)
            if dm:
                return dm.group(0), "high"

    # Fallback: any date anywhere (low confidence)
    for pat in DATE_PATTERNS:
        dm = re.search(pat, text, re.I)
        if dm:
            return dm.group(0), "low"
    return None, "low"

def normalize_date_to_iso(date_str: str | None):
    if not date_str:
        return None
    s = str(date_str).strip()
    # Handle fuzzy week-of dates: keep as-is (or you can map to an approximate date)
    if re.search(r"\bweek\s+of\b", s, re.I):
        return s
    try:
        dt = dtparser.parse(s, fuzzy=True, dayfirst=False)
        return dt.date().isoformat()
    except Exception:
        return s

In [150]:
def extract_ai_overview_text(result: dict) -> str:
    parts = []

    # 1️⃣ Main AI Overview (if present)
    blocks = result.get("ai_overview", {}).get("text_blocks", [])
    for b in blocks:
        if "snippet" in b:
            parts.append(b["snippet"])
        if "list" in b:
            for item in b["list"]:
                parts.append(item.get("snippet", ""))

    # 2️⃣ AI Overview inside related_questions (VERY COMMON)
    for rq in result.get("related_questions", []):
        if rq.get("type") == "ai_overview":
            for b in rq.get("text_blocks", []):
                if "snippet" in b:
                    parts.append(b["snippet"])
                if "list" in b:
                    for item in b["list"]:
                        parts.append(item.get("snippet", ""))

    return " ".join(parts).strip()


In [151]:
def extract_fallback_text(result):
    parts = []

    ab = result.get("answer_box", {})
    if isinstance(ab, dict):
        if ab.get("snippet"):
            parts.append(ab["snippet"])
        if ab.get("answer"):
            parts.append(str(ab["answer"]))

    for r in result.get("organic_results", [])[:3]:
        if r.get("snippet"):
            parts.append(r["snippet"])

    return " ".join(parts).strip()


## SerpAPI Key Setup

**SerpAPI** provides Google search results with **AI Overview** (Google's generated summaries).

### How to get your API key:
1. Go to [SerpAPI.com](https://serpapi.com)
2. Sign up for a free account (free tier: 100 searches/month)
3. Navigate to **Dashboard** → **API Key**
4. Copy your API key and replace `PUT_YOUR_SERPAPI_KEY_HERE` in the config cell below

**Cost**: Free tier gives 250 free searches/month. After that, pay-as-you-go ($5 per 10k searches).

### Rate Limiting:
- Current config: **1.2 seconds** between calls to be respectful
- Safety cap: **240 calls** per run (stays within free tier)
- Each movie = 1 API call

In [152]:
def serp_search(query: str):
    params = {
        "engine": "google",
        "q": query,
        "google_domain": "google.com",
        "hl": "en",
        "gl": "us",
        "location": "Austin, Texas, United States",
        "device": "desktop",
        "api_key": SERP_API_KEY,
    }
    r = requests.get("https://serpapi.com/search", params=params, timeout=30)
    r.raise_for_status()
    return r.json()

In [153]:
# Load Excel
df = pd.read_excel(INPUT_XLSX)

# Ensure output columns exist
for col in [COL_OTT_DATE, COL_CONFIDENCE, COL_EVIDENCE, COL_SOURCES]:
    if col not in df.columns:
        df[col] = ""

# ===== USER OPTIONS =====
print("📋 FILTER OPTIONS\n")

# Option 1: Select language
print("Available languages:", df[COL_LANGUAGE].unique().tolist())
FILTER_LANGUAGE = input("Enter language to filter (leave empty for all): ").strip()
if FILTER_LANGUAGE:
    FILTER_LANGUAGE = FILTER_LANGUAGE.lower()
    df_filtered = df[df[COL_LANGUAGE].str.lower() == FILTER_LANGUAGE]
else:
    df_filtered = df.copy()
print(f"✅ Filtered to {len(df_filtered)} rows with language '{FILTER_LANGUAGE or 'all'}'")

# Option 2: Select month range
MONTHS_INPUT = input("\nEnter number of months to look back (default 2): ").strip()
try:
    months_back = int(MONTHS_INPUT) if MONTHS_INPUT else 2
except ValueError:
    months_back = 1
    print(f"Invalid input, using default: {months_back} months")

print(f"✅ Looking back {months_back} months\n")

# Parse releaseDate for filtering recency
def safe_parse_date(x):
    if pd.isna(x) or x is None or str(x).strip()=="":
        return None
    try:
        return dtparser.parse(str(x), fuzzy=True).date()
    except Exception:
        return None

df_filtered["__releaseDateParsed__"] = df_filtered[COL_RELEASEDATE].apply(safe_parse_date)

cutoff = (datetime.now().date() - relativedelta(months=months_back))
recent_mask = df_filtered["__releaseDateParsed__"].apply(lambda d: d is not None and d >= cutoff)

# Missing OTT mask
missing_mask = df_filtered[COL_OTT_LIST].apply(is_missing_ott)

target = df_filtered[recent_mask & missing_mask].copy()
print("Total rows:", len(df))
print(f"Filtered by language '{FILTER_LANGUAGE or 'all'}': {len(df_filtered)}")
print(f"Recent (last {months_back} months):", int(recent_mask.sum()))
print(f"Recent + missing OTT: {len(target)}")


📋 FILTER OPTIONS

Available languages: ['Telugu', 'Hindi', 'Tamil', 'Malayalam']
✅ Filtered to 204 rows with language 'malayalam'
✅ Looking back 12 months

Total rows: 675
Filtered by language 'malayalam': 204
Recent (last 12 months): 204
Recent + missing OTT: 128


/tmp/ipykernel_20081/1780751510.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["__releaseDateParsed__"] = df_filtered[COL_RELEASEDATE].apply(safe_parse_date)


In [154]:
# # ✅ QUICK TEST: Single title (Akhanda 2) — uses 1 SerpAPI call

# def build_query(title: str, language: str | None):
#     lang = (language or "").strip()
#     if lang:
#         return f"{title} {lang} OTT release date streaming platform"
#     return f"{title} OTT release date streaming platform"

# # --- Test input ---
# test_title = "Akhanda 2: Thaandavam"
# test_language = "Telugu"   # or "" if you want

# q = build_query(test_title, test_language)
# print("Query:", q)

# try:
#     res = serp_search(q)  # <-- 1 SerpAPI call

#     ai_text = extract_ai_overview_text(res)
#     if not ai_text:
#         ai_text = extract_fallback_text(res)

#     print("\n--- AI OVERVIEW TEXT (first 800 chars) ---")
#     print((ai_text[:800] + " ...") if ai_text else "No ai_overview returned")

#     platforms = extract_platforms(ai_text) if ai_text else []
#     ott_date_raw, conf_date = extract_ott_date(ai_text) if ai_text else (None, "low")
#     ott_date = normalize_date_to_iso(ott_date_raw)

#     confidence = "high" if platforms and conf_date == "high" else ("medium" if platforms or ott_date else "low")

#     sources = []
#     for r in res.get("organic_results", [])[:3]:
#         link = r.get("link")
#         if link:
#             sources.append(link)

#     print("\n--- EXTRACTED ---")
#     print("Platforms:", platforms)
#     print("OTT date:", ott_date)
#     print("Confidence:", confidence)
#     print("Sources:", sources)

# except requests.HTTPError as e:
#     print("SerpAPI HTTPError:", e)
# except Exception as e:
#     print("Error:", e)


In [155]:
# # Save and verify the test result for Akhanda 2
# print("\n" + "="*60)
# print("SAVING TEST RESULT TO ENRICHMENT FILE")
# print("="*60)

# # Create a test dataframe with just the Akhanda 2 result
# test_movie = {
#     COL_TITLE: test_title,
#     COL_LANGUAGE: test_language,
#     COL_OTT_LIST: json.dumps(platforms, ensure_ascii=False) if platforms else "",
#     COL_OTT_DATE: ott_date or "",
#     COL_CONFIDENCE: confidence,
#     COL_EVIDENCE: ai_text[:500] if ai_text else "",
#     COL_SOURCES: json.dumps(sources, ensure_ascii=False) if sources else ""
# }

# # Save to enrichment file (append or create)
# enrichment_file = OUTPUT_XLSX
# test_df = pd.DataFrame([test_movie])
# test_df.to_excel(enrichment_file, index=False)
# print(f"\n✅ Test result saved to: {enrichment_file}")

# # Read back and verify
# print("\n" + "="*60)
# print("VERIFICATION - Reading back from file")
# print("="*60)
# verified_df = pd.read_excel(enrichment_file)
# print("\nSaved data:")
# for col in [COL_TITLE, COL_OTT_LIST, COL_OTT_DATE, COL_CONFIDENCE]:
#     print(f"  {col}: {verified_df.loc[0, col]}")

# print("\n✅ Akhanda 2 data successfully saved and verified!")

In [156]:
# ===== CHECK FOR NEW DATA =====
print("="*60)
print("CHECKING FOR NEW DATA TO ENRICH")
print("="*60)

if len(target) == 0:
    print("\n⚠️  No new data found to enrich!")
    print(f"   (Filtered: {FILTER_LANGUAGE or 'all'} | Last {months_back} months | Missing OTT info)")
    print("\n✅ Skipping enrichment. No cleanup needed.")
else:
    print(f"\n✅ Found {len(target)} movies to enrich")
    
    # ===== CLEANUP OLD ENRICHMENT FILE =====
    import os
    if os.path.exists(OUTPUT_XLSX):
        os.remove(OUTPUT_XLSX)
        print(f"🗑️  Cleaned up old enrichment file: {OUTPUT_XLSX}")
    
    # ===== BULK ENRICHMENT =====
    print("\n" + "="*60)
    print("STARTING BULK ENRICHMENT")
    print("="*60 + "\n")
    
    enriched_rows = []
    
    def build_query(title: str, language: str | None):
        lang = (language or "").strip()
        if lang:
            return f"{title} {lang} OTT Release streaming platform"
        return f"{title} OTT Release streaming platform"
    
    calls = 0
    for idx, row in target.iterrows():
        if calls >= MAX_CALLS:
            print(f"\nReached MAX_CALLS={MAX_CALLS}. Stopping to avoid subscription limit.")
            break
    
        title = str(row.get(COL_TITLE, "")).strip()
        language = str(row.get(COL_LANGUAGE, "")).strip() if COL_LANGUAGE in df.columns else ""
        if not title:
            continue
    
        q = build_query(title, language)
        print(f"\n[{calls+1}] Enriching: {title} ({language})")
        print(f"    Query: {q}")
        
        try:
            time.sleep(SLEEP_SECONDS)
            res = serp_search(q)
            calls += 1
            
            # Extract text
            ai_text = extract_ai_overview_text(res)
            if not ai_text:
                ai_text = extract_fallback_text(res)
            
            # Extract data
            platforms = extract_platforms(ai_text) if ai_text else []
            ott_date_raw, conf_date = extract_ott_date(ai_text) if ai_text else (None, "low")
            ott_date = normalize_date_to_iso(ott_date_raw)
            
            confidence = "high" if platforms and conf_date == "high" else ("medium" if platforms or ott_date else "low")
            
            sources = []
            for r in res.get("organic_results", [])[:3]:
                link = r.get("link")
                if link:
                    sources.append(link)
            
            # Store result
            enriched_rows.append({
                COL_TITLE: title,
                COL_LANGUAGE: language,
                COL_OTT_LIST: json.dumps(platforms, ensure_ascii=False) if platforms else "",
                COL_OTT_DATE: ott_date or "",
                COL_CONFIDENCE: confidence,
                COL_EVIDENCE: ai_text[:500] if ai_text else "",
                COL_SOURCES: json.dumps(sources, ensure_ascii=False) if sources else ""
            })
            
            print(f"    ✓ Platforms: {platforms}")
            print(f"    ✓ OTT Date: {ott_date}")
            print(f"    ✓ Confidence: {confidence}")
        
        except requests.HTTPError as e:
            print(f"    ✗ SerpAPI HTTPError: {e}")
        except Exception as e:
            print(f"    ✗ Error: {e}")
    
    # Save enriched data
    if enriched_rows:
        enriched_df = pd.DataFrame(enriched_rows)
        enriched_df.to_excel(OUTPUT_XLSX, index=False)
        print(f"\n✅ Enrichment complete! Saved {len(enriched_rows)} rows to: {OUTPUT_XLSX}")
        print(f"   API Calls used: {calls}/{MAX_CALLS}")
    else:
        print("\n⚠️  No enriched data to save.")

CHECKING FOR NEW DATA TO ENRICH

✅ Found 128 movies to enrich

STARTING BULK ENRICHMENT


[1] Enriching: Lokah Chapter 1: Chandra (Malayalam)
    Query: Lokah Chapter 1: Chandra Malayalam OTT Release streaming platform
    ✓ Platforms: ['Disney+ Hotstar']
    ✓ OTT Date: None
    ✓ Confidence: medium

[2] Enriching: Bha. Bha. Ba. (Malayalam)
    Query: Bha. Bha. Ba. Malayalam OTT Release streaming platform
    ✓ Platforms: []
    ✓ OTT Date: 2024-11-29
    ✓ Confidence: medium

[3] Enriching: Vilaayath Budha (Malayalam)
    Query: Vilaayath Budha Malayalam OTT Release streaming platform
    ✓ Platforms: []
    ✓ OTT Date: 2024-11-29
    ✓ Confidence: medium

[4] Enriching: Haal (Malayalam)
    Query: Haal Malayalam OTT Release streaming platform
    ✓ Platforms: []
    ✓ OTT Date: 2024-11-29
    ✓ Confidence: medium

[5] Enriching: Sarvam Maya (Malayalam)
    Query: Sarvam Maya Malayalam OTT Release streaming platform
    ✓ Platforms: ['Netflix', 'Prime Video']
    ✓ OTT Date: 2024-11-

In [157]:
# Merge enriched data back to master file
print("\n" + "="*60)
print("UPDATING MASTER FILE WITH ENRICHED DATA")
print("="*60)

# Read enriched file
enriched_df = pd.read_excel(OUTPUT_XLSX)
print(f"\n📖 Loaded enriched file: {len(enriched_df)} rows")

# Read original master
master_df = pd.read_excel(INPUT_XLSX)
print(f"📖 Loaded master file: {len(master_df)} rows")

# Merge: match by title (case-insensitive) and optionally language
merged_count = 0
updated_ott_count = 0
updated_date_count = 0

for idx_enriched, enriched_row in enriched_df.iterrows():
    enriched_title = str(enriched_row.get(COL_TITLE, "")).strip().lower()
    enriched_language = str(enriched_row.get(COL_LANGUAGE, "")).strip().lower()
    enriched_ott = enriched_row.get(COL_OTT_LIST, "")
    enriched_date = enriched_row.get(COL_OTT_DATE, "")
    
    if not enriched_title:
        continue
    
    # Find matching row in master (match by title, prefer language match)
    matched_idx = None
    for idx_master, master_row in master_df.iterrows():
        master_title = str(master_row.get(COL_TITLE, "")).strip().lower()
        master_language = str(master_row.get(COL_LANGUAGE, "")).strip().lower()
        
        # Exact title + language match
        if enriched_title == master_title and enriched_language == master_language:
            matched_idx = idx_master
            break
        # Title-only match (fallback)
        elif enriched_title == master_title and matched_idx is None:
            matched_idx = idx_master
    
    if matched_idx is not None:
        # Update OTT List
        if enriched_ott and str(enriched_ott).strip() not in ["", "[]", "nan"]:
            master_df.at[matched_idx, COL_OTT_LIST] = enriched_ott
            updated_ott_count += 1
        
        # Update OTT Release Date
        if enriched_date and str(enriched_date).strip() not in ["", "nan"]:
            master_df.at[matched_idx, COL_OTT_DATE] = enriched_date
            updated_date_count += 1
        
        merged_count += 1

print(f"\n✅ Matched and updated {merged_count} movies")
print(f"   - OTT Lists updated: {updated_ott_count}")
print(f"   - OTT Release Dates updated: {updated_date_count}")

# Save updated master
master_df.to_excel(INPUT_XLSX, index=False)
print(f"\n✅ Updated master file saved: {INPUT_XLSX}")

# Display sample of updated rows
print("\n" + "="*60)
print("SAMPLE OF UPDATED ROWS")
print("="*60)
for idx in range(min(3, len(enriched_df))):
    title = enriched_df.loc[idx, COL_TITLE]
    ott_list = enriched_df.loc[idx, COL_OTT_LIST]
    ott_date = enriched_df.loc[idx, COL_OTT_DATE]
    print(f"\n✓ {title}")
    print(f"  OTT Platforms: {ott_list}")
    print(f"  OTT Release Date: {ott_date}")



UPDATING MASTER FILE WITH ENRICHED DATA

📖 Loaded enriched file: 128 rows
📖 Loaded master file: 675 rows

✅ Matched and updated 128 movies
   - OTT Lists updated: 50
   - OTT Release Dates updated: 102

✅ Updated master file saved: movies_master.xlsx

SAMPLE OF UPDATED ROWS

✓ Lokah Chapter 1: Chandra
  OTT Platforms: ["Disney+ Hotstar"]
  OTT Release Date: nan

✓ Bha. Bha. Ba.
  OTT Platforms: nan
  OTT Release Date: 2024-11-29

✓ Vilaayath Budha
  OTT Platforms: nan
  OTT Release Date: 2024-11-29


In [158]:
# Verify the updates by checking a few sample rows from the enriched data
print("=== VERIFICATION OF ENRICHED DATA ===\n")

# Display columns to verify
verify_cols = [COL_TITLE, COL_OTT_LIST, COL_OTT_DATE, COL_CONFIDENCE, COL_SOURCES]

# Show a sample of enriched rows (rows that were in target)
if len(target) > 0:
    print(f"Sample of {min(5, len(target))} enriched rows:\n")
    sample_indices = target.index[:5]
    
    for idx in sample_indices:
        print(f"\n--- {df.loc[idx, COL_TITLE]} ---")
        print(f"OTT Platforms: {df.loc[idx, COL_OTT_LIST]}")
        print(f"OTT Release Date: {df.loc[idx, COL_OTT_DATE]}")
        print(f"Confidence: {df.loc[idx, COL_CONFIDENCE]}")
        print(f"Sources: {df.loc[idx, COL_SOURCES][:100]}..." if len(str(df.loc[idx, COL_SOURCES])) > 100 else f"Sources: {df.loc[idx, COL_SOURCES]}")

    # Summary statistics
    print("\n\n=== SUMMARY STATISTICS ===")
    print(f"Total rows processed: {len(target)}")
    print(f"Rows with OTT platforms found: {df.loc[target.index, COL_OTT_LIST].apply(lambda x: x not in ['[]', '', None] and not pd.isna(x)).sum()}")
    print(f"Rows with OTT dates found: {df.loc[target.index, COL_OTT_DATE].apply(lambda x: x not in ['', None] and not pd.isna(x)).sum()}")
    print(f"High confidence: {(df.loc[target.index, COL_CONFIDENCE] == 'high').sum()}")
    print(f"Medium confidence: {(df.loc[target.index, COL_CONFIDENCE] == 'medium').sum()}")
    print(f"Low confidence: {(df.loc[target.index, COL_CONFIDENCE] == 'low').sum()}")
else:
    print("No rows were enriched (target was empty)")

=== VERIFICATION OF ENRICHED DATA ===

Sample of 5 enriched rows:


--- Lokah Chapter 1: Chandra ---
OTT Platforms: []
OTT Release Date: nan
Confidence: 
Sources: 

--- Bha. Bha. Ba. ---
OTT Platforms: []
OTT Release Date: nan
Confidence: 
Sources: 

--- Vilaayath Budha ---
OTT Platforms: []
OTT Release Date: nan
Confidence: 
Sources: 

--- Haal ---
OTT Platforms: []
OTT Release Date: nan
Confidence: 
Sources: 

--- Sarvam Maya ---
OTT Platforms: []
OTT Release Date: nan
Confidence: 
Sources: 


=== SUMMARY STATISTICS ===
Total rows processed: 128
Rows with OTT platforms found: 0
Rows with OTT dates found: 0
High confidence: 0
Medium confidence: 0
Low confidence: 0
